<a href="https://colab.research.google.com/github/Sagaahmedd/HousePricePrediction/blob/main/HousePrice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning Regression Comparison
## Comparing Regression Techniques

This notebook implements and compares different regression algorithms on housing price data:
1. Linear regression
2. Ridge regression
3. Lasso regression
4. ElasticNet
5. KNN
6. XGboost (Extra)

# 1. Import Libraries

In [389]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import xgboost as xgb
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

All libraries imported successfully!


# 2. Load and Prepare Data

In [390]:
# Load the datasets
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

print(f"Training data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")

# Separate training data
X_train =train_df.drop('SalePrice', axis=1)
y_train =train_df['SalePrice']
#test data
X_test_full= test_df


Training data shape: (1460, 81)
Test data shape: (1459, 80)


In [391]:
#split to categorical and numerical
cat_cols = X_train.select_dtypes(include="object").columns
num_cols = X_train.select_dtypes(exclude="object").columns

In [392]:
#Known ordered patterns
patterns = [
    ["Po","Fa","TA","Gd","Ex"],
    ["None","Po","Fa","TA","Gd","Ex"],
    ["No","Mn","Av","Gd"],
    ["None","No","Mn","Av","Gd"],
    ["Unf","RFn","Fin"],
    ["None","Unf","RFn","Fin"],
    ["Unf","LwQ","Rec","BLQ","ALQ","GLQ"],
    ["None","Unf","LwQ","Rec","BLQ","ALQ","GLQ"],
    ["MnWw","GdWo","MnPrv","GdPrv"],
    ["None","MnWw","GdWo","MnPrv","GdPrv"]
]
ordinal_cols = []
ordinal_categories = [] # New list to store categories for each ordinal column

for col in cat_cols:
    values = set(X[col].dropna().unique())

    for pattern in patterns:
        if values.issubset(set(pattern)):
            ordinal_cols.append(col)
            ordinal_categories.append(pattern) # Store the matched pattern
            break

# remaining categorical data=nominal
nominal_cols = [col for col in cat_cols if col not in ordinal_cols]

print("Ordinal:", ordinal_cols)
print("Nominal:", nominal_cols)
print("Ordinal Categories:", ordinal_categories)

Ordinal: ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Fence']
Nominal: ['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation', 'Heating', 'CentralAir', 'Electrical', 'Functional', 'GarageType', 'PavedDrive', 'MiscFeature', 'SaleType', 'SaleCondition']
Ordinal Categories: [['Po', 'Fa', 'TA', 'Gd', 'Ex'], ['Po', 'Fa', 'TA', 'Gd', 'Ex'], ['Po', 'Fa', 'TA', 'Gd', 'Ex'], ['Po', 'Fa', 'TA', 'Gd', 'Ex'], ['No', 'Mn', 'Av', 'Gd'], ['Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'], ['Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'], ['Po', 'Fa', 'TA', 'Gd', 'Ex'], ['Po', 'Fa', 'TA', 'Gd', 'Ex'], ['Po', 'Fa', 'TA', 'Gd', 'Ex'], ['Unf', 'RFn', 'Fin'], ['Po', 'Fa', 'TA', 

In [393]:
# 1. Drop columns with >40% missing
cols_to_drop = X_train.columns[X_train.isnull().mean() > 0.4].tolist()
X_train, X_test_full = [df.drop(columns=cols_to_drop) for df in [X_train, X_test_full]]
print(f"Dropped: {cols_to_drop}")

Dropped: ['Alley', 'MasVnrType', 'FireplaceQu', 'PoolQC', 'Fence', 'MiscFeature']


In [394]:
# Update column lists
num_cols = [c for c in num_cols if c not in cols_to_drop]
cat_cols = [c for c in cat_cols if c not in cols_to_drop]
ordinal_cols = [c for c in ordinal_cols if c not in cols_to_drop]
nominal_cols = [c for c in nominal_cols if c not in cols_to_drop]

#keep ordinal_categories in sync with ordinal_cols
ordinal_categories = [cat for col, cat in zip(ordinal_cols, ordinal_categories) if col in X_train.columns]
ordinal_cols = [col for col in ordinal_cols if col in X_train.columns]

# 2. Apply Log-Transform

> log-transform skewed targets and features using (log1p)



In [395]:
skewed_cols = X_train[num_cols].skew()[lambda s: s.abs() > 0.75].index.tolist()
for df in [X_train, X_test_full]:
    df[skewed_cols] = np.log1p(df[skewed_cols])
y_train = np.log1p(y_train)
print(f"Log-transformed {len(skewed_cols)} features + target ✓")


Log-transformed 21 features + target ✓


#3. Pipeline

> fill missing values with median for numerical data

> fill missing values with mode for categorical data

> one-hot-encoding for categorical features

> ordinal encoding for ordered features

> Standarization for numerical features










In [396]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), num_cols),

    ('ordinal', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(
            categories=ordinal_categories,
            handle_unknown='use_encoded_value',
            unknown_value=-1
        ))
    ]), ordinal_cols),

    ('nominal', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
    ]), nominal_cols)
])

X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled = preprocessor.transform(X_test_full)
print("Preprocessing done ✓")

Preprocessing done ✓


In [397]:
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(X_train_scaled, y_train, test_size=0.2, random_state=42)

# 4. Train Models

### 1. Linear Regression (OLS)
> main model





In [398]:
print("Training Linear Regression...")

lr_model = LinearRegression()
lr_model.fit(X_tr, y_tr)

# Metrics
lr_pred_train = lr_model.predict(X_tr)
lr_pred_val = lr_model.predict(X_val)

print(f"\n✓ Linear Regression Results:")
print(f"  Train RMSE: {np.sqrt(mean_squared_error(y_tr, lr_pred_train)):.4f}")  #error size
print(f"  Val RMSE:   {np.sqrt(mean_squared_error(y_val, lr_pred_val)):.4f}")
print(f"  Train R²:   {r2_score(y_tr, lr_pred_train):.4f}")
print(f"  Val R²:     {r2_score(y_val, lr_pred_val):.4f}")

Training Linear Regression...

✓ Linear Regression Results:
  Train RMSE: 0.0945
  Val RMSE:   0.1227
  Train R²:   0.9415
  Val R²:     0.9194


In [399]:
#Train R² = 0.94 → model explains 94% of variance in training data
#Val R² = 0.92 → model explains 92% of variance on unseen data

## 2. Lasso regression (L1 — also performs feature selection)

> top-10 most important features (from Lasso coefficients).



## 3. Ridge regression (L2 regularization)

## 4. ElasticNet (L1 + L2 mix)

## 5. KNN (non-parametric baseline)

## 6. XGBoost (Extra)

In [400]:
print("Training XGBoost...")

xgb_model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=1
)

xgb_model.fit(X_tr, y_tr,eval_set=[(X_val, y_val)],verbose=False)

# Predictions
xgb_pred_train = xgb_model.predict(X_tr)
xgb_pred_val = xgb_model.predict(X_val)

# Calculate metrics
xgb_train_rmse = np.sqrt(mean_squared_error(y_tr, xgb_pred_train))
xgb_val_rmse = np.sqrt(mean_squared_error(y_val, xgb_pred_val))
xgb_train_r2 = r2_score(y_tr, xgb_pred_train)
xgb_val_r2 = r2_score(y_val, xgb_pred_val)

print(f"\n✓ XGBoost Results:")
print(f"  Train RMSE: ${xgb_train_rmse:,.2f}")
print(f"  Val RMSE: ${xgb_val_rmse:,.2f}")
print(f"  Train R²: {xgb_train_r2:.4f}")
print(f"  Val R²: {xgb_val_r2:.4f}")

Training XGBoost...

✓ XGBoost Results:
  Train RMSE: $0.01
  Val RMSE: $0.14
  Train R²: 0.9991
  Val R²: 0.8945


In [401]:
# Display nice summary table
summary_df = df_comparison.copy()
summary_df['Val RMSE'] = summary_df['Val RMSE'].apply(lambda x: f"${x:,.2f}")
summary_df['Train RMSE'] = summary_df['Train RMSE'].apply(lambda x: f"${x:,.2f}")
summary_df['Val MAE'] = summary_df['Val MAE'].apply(lambda x: f"${x:,.2f}")
summary_df['Train MAE'] = summary_df['Train MAE'].apply(lambda x: f"${x:,.2f}")
summary_df['Val R2'] = summary_df['Val R2'].apply(lambda x: f"{x:.4f}")
summary_df['Train R2'] = summary_df['Train R2'].apply(lambda x: f"{x:.4f}")

from IPython.display import display, HTML

# Style the dataframe
styled_df = summary_df.style.set_properties(**{
    'text-align': 'center',
    'font-size': '12pt'
}).set_table_styles([
    {
        'selector': 'th',
        'props': [('font-size', '14pt'), ('font-weight', 'bold'), ('text-align', 'center')]
    }
])

display(styled_df)


,Model,Train RMSE,Val RMSE,Train MAE,Val MAE,Train R2,Val R2
0,XGBoost,"$2,327.45","$27,449.09","$1,722.61","$17,430.36",0.9991,0.9018
1,Linear Regression,"$33,920.14","$36,836.91","$21,066.67","$22,975.86",0.8071,0.8231


## 5. Model Comparison

>Data visualisation

> Bar chart comparing CV RMSE across the 5 models + table of top-10 most important features (from Lasso coefficients).

> Residual & Q-Q plot for best model only



## 6. Visualize Model Performance

## 8. Summary Table

## 9. Gradio Interface

In [402]:
#Gradio interface
import gradio as gr

# Calculate average percentage error on validation set
percentage_errors = np.abs((y_val -xgb_top10_pred_val) / y_val) * 100
avg_percentage_error = np.mean(percentage_errors)
avg_accuracy_percentage = 100 - avg_percentage_error

# Prediction function
def predict_house_price(*args):
  # Create dataframe from user inputs
  input_data = pd.DataFrame([args], columns=top_10_xgb_features)

  # Make prediction
  prediction = xgb_top10.predict(input_data)[0]

  # Calculate prediction accuracy% using MAPE(Mean Absolute Percentage Error)
  expected_error_percentage = (xgb_top10_val_mae / prediction) * 100
  prediction_accuracy = 100 - expected_error_percentage

  # Ensure accuracy is within reasonable bounds
  prediction_accuracy = max(0, min(100, prediction_accuracy))

  # Format the predicted price output with accuracy
  price_output = f"${prediction:,.2f}\n\n🎯 Prediction Accuracy: {prediction_accuracy:.1f}%"

  # Format detailed accuracy text
  accuracy_text = f"""

🎯 **This Prediction Accuracy:** {prediction_accuracy:.1f}%
- Expected error range: ±${xgb_top10_val_mae:,.2f} (±{expected_error_percentage:.1f}%)
- Predicted price range: ${prediction - xgb_top10_val_mae:,.2f} → ${prediction + xgb_top10_val_mae:,.2f}

✅ **Confidence Level:** {"HIGH" if prediction_accuracy >= 90 else "MEDIUM" if prediction_accuracy >= 80 else "MODERATE"}
"""
  return price_output, accuracy_text


In [403]:
#Create gradio interface

#Feature name mapping for user-friendly display
feature_labels = {
    'OverallQual': '⭐ Overall Quality (1-10)',
    'GarageCars': '🚗 Garage Capacity (Cars)',
    'GrLivArea': '🏠 Living Area (sq ft)',
    'BsmtFinSF1': '🔨 Finished Basement Area (sq ft)',
    '2ndFlrSF': '⬆️ Second Floor Area (sq ft)',
    'PoolArea': '🏊 Pool Area (sq ft)',
    'Fireplaces': '🔥 Number of Fireplaces',
    'TotalBsmtSF': '📦 Total Basement Area (sq ft)',
    'YearBuilt': '📅 Year Built',
    'KitchenAbvGr': '👨‍🍳 Kitchens Above Ground'
}
# Determine dataset for slider initialization
slider_data = (X_train_top10)

# Create sliders with friendly labels
sliders = [
    gr.Slider(
        minimum=float(slider_data[f].min()),
        maximum=float(slider_data[f].max()),
        value=float(slider_data[f].median()),
        label=feature_labels.get(f, f),
        info=f"Range: {slider_data[f].min():.0f} - {slider_data[f].max():.0f}"
    ) for f in top_10_xgb_features
]

with gr.Blocks(theme=gr.themes.Monochrome(), title="House Price Predictor") as demo:

    gr.Markdown("""
<div style="text-align: center;">
<h1 style="font-size: 50px; margin-bottom: 10px;">House Price Prediction System 🏡</h1>
<h3 style="font-size: 24px; margin-top: 0;">Powered by XGBoost Machine Learning Model</h3>
</div>
""")

    with gr.Row():
        with gr.Column():
            gr.Markdown("## House Features")
            # Render the sliders
            for slider in sliders:
                slider.render()

        with gr.Column():
            gr.Markdown("## 💰 Prediction Results")
            predict_btn = gr.Button("Predict Price", variant="primary", size="lg")
            predicted_price = gr.Textbox("Predicted House Price & Accuracy", interactive=False, lines=3)
            accuracy_output = gr.Markdown("*Click 'Predict Price' to see detailed accuracy metrics*")

    gr.Markdown("""
    ---
    ### 📖 How to Use
    Adjust sliders → Predict → Review accuracy
    """)

    # Connect prediction function
    predict_btn.click(predict_house_price, inputs=sliders, outputs=[predicted_price, accuracy_output])


In [404]:
#LAUNCH INTERFACE

print("=" * 60)
print("LAUNCHING GRADIO INTERFACE...")
print("=" * 60)

demo.launch(share=True)

LAUNCHING GRADIO INTERFACE...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://83cfcf045e3dc0441a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
